# Do the Groups Predict Churn?

**The business question.** A telecom company is losing about a quarter of its customers, and the retention team has a fixed budget. Treating all customers alike wastes it. If customers fall into distinct kinds, each kind can get a different conversation. So: **do the groups we find in how customers pay and how long they stay turn out to differ in who leaves?**

Here is the twist. We cluster the customers **without looking at churn at all**. Then we look at churn and see whether the groups we found differ in it. If they do, the segmentation has found something that matters to the business, not just something that looks tidy on a chart.

**How this notebook works.** One part, five steps:

1. Load and look
2. Run the model
3. Read the output in plain English
4. Check against JMP
5. Say it to a manager

Copilot is switched on. Check anything it writes against the JMP result. There is a Stretch space at the end.

The data is a real-style telecom customer file, with 7,043 customers.

## Set up

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

pd.set_option("display.width", 140)
print("Libraries loaded.")

### Settings used throughout this module

In [ ]:
N_INIT = 25      # k-means tries 25 different starting points and keeps the best one
SEED = 1         # fixes the random choices so a re-run gives the same answer
# Scaling: StandardScaler on the clustering columns, always.

---
## Step 1. Load and look

In [ ]:
telco = pd.read_excel("../data/Telco-Customer-Churn.xlsx", sheet_name="Telco-Customer-Churn")
print("customers:", telco.shape[0], "  columns:", telco.shape[1])
print(telco[["tenure", "MonthlyCharges", "Contract", "InternetService", "Churn"]].head().to_string(index=False))
print()
print("Overall churn rate:", f"{(telco['Churn'] == 'Yes').mean():.1%}")

We cluster on three columns that describe how a customer is tied to the company:

- `tenure`: months as a customer
- `MonthlyCharges`: the monthly bill
- contract level: month-to-month, one year or two year

The contract is text, so we turn it into a number that keeps its order: 0 for month-to-month, 1 for one year, 2 for two year. `Churn` is **held back**. The clustering must never see it.

In [ ]:
telco["ContractLevel"] = telco["Contract"].map({"Month-to-month": 0, "One year": 1, "Two year": 2})
CLUSTER_COLUMNS = ["tenure", "MonthlyCharges", "ContractLevel"]
print(telco[CLUSTER_COLUMNS].describe().round(2).to_string())

**What to look for.** The three columns are measured in months, money and a 0 to 2 code. They live on very different scales, which the next step has to deal with.

## Step 2. Run the model

In [ ]:
scaled = StandardScaler().fit_transform(telco[CLUSTER_COLUMNS])    # put every column on the same scale

rows = []
for k in range(2, 8):
    km_k = KMeans(n_clusters=k, n_init=N_INIT, random_state=SEED).fit(scaled)
    rows.append({"k": k, "inertia": km_k.inertia_, "silhouette": silhouette_score(scaled, km_k.labels_)})
sweep = pd.DataFrame(rows)
print(sweep.round({"inertia": 1, "silhouette": 3}).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(sweep["k"], sweep["inertia"], marker="o")
ax[0].set_title("Inertia")
ax[1].plot(sweep["k"], sweep["silhouette"], marker="o", color="tab:orange")
ax[1].set_title("Silhouette")
ax[1].set_ylim(0, 0.6)
for a in ax:
    a.set_xlabel("k, the number of groups")
plt.tight_layout()
plt.show()

**What to look for.** The silhouette barely moves as k changes. When that happens the data is not choosing k for you, so **k is a business decision**. We use **k = 4**: the retention team can write four different retention conversations, and four groups keeps each one big enough to act on.

In [ ]:
K = 4
km = KMeans(n_clusters=K, n_init=N_INIT, random_state=SEED).fit(scaled)
telco["Group"] = km.labels_
print("Customers in each group:")
print(telco["Group"].value_counts().sort_index().to_string())

## Step 3. Read the output in plain English

Group numbers are arbitrary, so read the averages. The profile also shows the share of customers on fibre broadband. Fibre was **not** a clustering column, so any pattern in it is a bonus finding.

In [ ]:
telco["Fibre"] = (telco["InternetService"] == "Fiber optic").astype(int)
telco["ChurnFlag"] = (telco["Churn"] == "Yes").astype(int)

profile = telco.groupby("Group").agg(
    Customers=("customerID", "size"),
    Tenure=("tenure", "mean"),
    MonthlyCharges=("MonthlyCharges", "mean"),
    ContractLevel=("ContractLevel", "mean"),
    FibreShare=("Fibre", "mean"),
)
print(profile.round(2).to_string())

**Name each group** in a few words that say what kind of customer it is. For example, "long-standing, low bill" is better than "Group 2".

**Your answer:**

Now the test. Churn was never used to build the groups. Does it differ between them?

In [ ]:
churn = telco.groupby("Group").agg(Customers=("customerID", "size"), ChurnRate=("ChurnFlag", "mean"),
                                    Churners=("ChurnFlag", "sum"))
churn["ShareOfAllChurners"] = churn["Churners"] / churn["Churners"].sum()
churn["ChurnRate"] = (churn["ChurnRate"] * 100).round(1)
churn["ShareOfAllChurners"] = (churn["ShareOfAllChurners"] * 100).round(1)
print(churn.to_string())

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(churn.index.astype(str), churn["ChurnRate"], color="#1A4C3B")
ax.axhline((telco["ChurnFlag"].mean() * 100), color="#E07B39", linestyle="--", label="Overall churn rate")
ax.set_xlabel("Group (numbers are arbitrary)")
ax.set_ylabel("Churn rate, %")
ax.legend()
plt.tight_layout()
plt.show()

**What to look for.** The spread in churn rate between groups, and where most of the people who actually leave are found. A retention budget should go where the leavers are, not where the customers are.

**Your answer.** Which group would you put the retention budget into first, and why?

## Step 4. Check against JMP

Export the table for JMP. The file goes into an `exports` folder next to this notebook.

In [ ]:
Path("exports").mkdir(exist_ok=True)
export = telco[["customerID", "tenure", "MonthlyCharges", "ContractLevel", "Churn"]].copy()
export.insert(0, "CustomerNumber", np.arange(1, len(export) + 1))
export["ChurnFlag"] = telco["ChurnFlag"]
export.to_csv("exports/telco_cluster_columns.csv", index=False)
print("Saved exports/telco_cluster_columns.csv with", len(export), "rows")

In JMP, run K Means clustering on `tenure`, `MonthlyCharges` and `ContractLevel` with k = 4, using the settings in the Activity Workbook, Clustering in JMP. Then find the churn rate in each JMP group. Group sizes will differ a little between JMP and Python and the labels will not match, which is expected. The pattern in churn should be recognisable.

In [ ]:
jmp_sizes = []          # type the group sizes JMP gives you, largest first, for example [900, 800, 700, 600]

python_sizes = sorted(telco["Group"].value_counts().tolist(), reverse=True)
if len(jmp_sizes) == K:
    check = pd.DataFrame({"Python": python_sizes, "JMP": jmp_sizes})
    check["Difference"] = check["Python"] - check["JMP"]
    print(check.to_string(index=False))
else:
    print("Python group sizes, largest first:", python_sizes)
    print("Add the four JMP sizes to jmp_sizes above and run this cell again.")

### Does the answer depend on k?

Because k was our choice, check that the churn finding does not depend on it. The loop below clusters again with 3, 4 and 5 groups and reports the lowest and highest churn rate across the groups.

In [ ]:
rows = []
for k in (3, 4, 5):
    labels = KMeans(n_clusters=k, n_init=N_INIT, random_state=SEED).fit(scaled).labels_
    rates = telco.groupby(labels)["ChurnFlag"].mean() * 100
    rows.append({"k": k, "lowest churn rate %": rates.min(), "highest churn rate %": rates.max()})
print(pd.DataFrame(rows).round(1).to_string(index=False))

## Step 5. Say it to a manager

Two sentences, to the head of retention. Say what the groups are, how much churn differs between them, and one thing the groups tell you that a single churn-probability score per customer would not.

**Your answer:**

**Ask Copilot.** Try this prompt, then check it against your tables:

> For the dataframe `telco` with a column `Group`, show the churn rate for each combination of Group and Contract. Is the contract type or the group doing more of the work?

Which of the two explains more of the difference in churn? How can you tell?

---
# Stretch

Quick finishers: choose one and use Copilot to write the code. Check every number.

- Fit a logistic regression for churn using only the three clustering columns, then compare its test-set AUC with how well the four groups separate churn. What does each approach tell the retention team that the other does not?
- Add `PaperlessBilling` or `PaymentMethod` to the profile table. Does it vary between groups even though it was never a clustering column?

In [ ]:
# Your space.